In [1]:
import pandas as pd
import numpy as np

# Генерируем датасет

In [2]:
dataset = {
    "еда": [
        "кофе с собой", "обед в столовой", "продукты в магните", "пицца с доставкой", 
        "шаурма", "пятерочка", "перекус в макдаке", "покупка хлеба", "суши", 
        "заказ еды на дом", "вино и сыр", "сходили в ресторан", "блины", "купил яблоки", "продукты на неделю",
        "взял мясо и курицу", "купил молоко", "поел в кафе", "заказал пиццу", "фрукты и овощи"
    ],
    "транспорт": [
        "такси до дома", "метро", "бензин", "каршеринг", "билет на автобус", 
        "яндекс такси", "заправка", "парковка", "мойка машины", "поезд в питер", 
        "самокат", "проездной", "оплата такси", "ремонт машины", "поездка",
        "оплата билета", "оплата абонемента на проезд", "каршеринг", "поезд до дома", "грузовое такси"
    ],
    "развлечения": [
        "билет в кино", "подписка на музыку", "концерт", "игра в стим", "поход в бар", 
        "боулинг", "театр", "книга", "кинотеатр", "нетфликс", "подписка на яндекс плюс", 
        "квест", "зоопарк", "билет на футбол", "настольные игры",
        "рыбалка с друзьями", "аренда коньков", "билет на баскетбол", "компьютерный клуб", "ночной клуб"
    ],
    "здоровье": [
        "аптека", "витамины", "визит к врачу", "стоматолог", "анализы", 
        "спортзал", "фитнес", "массаж", "покупка лекарств", "прием у терапевта", 
        "бассейн", "йога", "медицинская страховка", "таблетки от головы", "консультация",
        "дмс", "баскетбол", "бег", "медосмотр", "операция"
    ],
    "коммунальные": [
        "жкх", "квартплата", "интернет", "мобильная связь", "электричество", 
        "вода", "газ", "оплата квартиры", "коммуналка", "домашний интернет", 
        "пополнение телефона", "отопление", "вывоз мусора", "капремонт", "оплата связи",
        "оплата счётчиков", "свет", "ремонт в ванне", "ремонт на кухне", "ремонт в комнате"
    ],
    "прочее": [
        "wildberries", "ozon", "одежда", "кроссовки", "техника", 
        "подарок другу", "косметика", "бытовая химия", "новая куртка", "наушники", 
        "покупка книги", "товары для дома", "шампунь", "покупка посуды", "заказ на маркетплейсе",
        "ручка", "тетради", "покупка лампы", "покупка компьютера", "покупка ковра"
    ]
}

In [3]:
texts = []
labels = []

for category, examples in dataset.items():
    for example in examples:
        texts.append(example)
        labels.append(category)

print(f"Всего примеров: {len(texts)}")
print(f"Примеры текстов: {texts[:5]}")
print(f"Примеры категорий: {labels[:5]}")

Всего примеров: 120
Примеры текстов: ['кофе с собой', 'обед в столовой', 'продукты в магните', 'пицца с доставкой', 'шаурма']
Примеры категорий: ['еда', 'еда', 'еда', 'еда', 'еда']


# Обучение модели

In [11]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=8, stratify=labels
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5))),
    ('clf', LogisticRegression(max_iter=1000, random_state=8, C=10.0))
])

cv_scores = cross_val_score(pipeline, texts, labels, cv=5)
print(f"Кросс-валидация: {cv_scores}")
print(f"Средняя точность по кросс-валидации: {np.mean(cv_scores):.4f}\n")

Кросс-валидация: [0.5        0.29166667 0.41666667 0.5        0.70833333]
Средняя точность по кросс-валидации: 0.4833



## Проверка на своих примерах

In [28]:
pipeline.fit(texts, labels)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5))),
                ('clf',
                 LogisticRegression(C=10.0, max_iter=1000, random_state=8))])

In [29]:
def predict_expense(description):
    predicted_category = pipeline.predict([description])[0]
    probabilities = pipeline.predict_proba([description])[0]
    confidence = max(probabilities)
    
    print(f"Трата: '{description}'")
    print(f"-> Категория: {predicted_category.upper()}")
    print(f"-> Уверенность: {confidence:.2f}\n")

predict_expense("такси до работы")
predict_expense("купил новые кроссовки")
predict_expense("оплата света за месяц")
predict_expense("сходили в кино на выходных")
predict_expense("заказал роллы на ужин")
predict_expense("что-то непонятное")

Трата: 'такси до работы'
-> Категория: ТРАНСПОРТ
-> Уверенность: 0.82

Трата: 'купил новые кроссовки'
-> Категория: ПРОЧЕЕ
-> Уверенность: 0.58

Трата: 'оплата света за месяц'
-> Категория: КОММУНАЛЬНЫЕ
-> Уверенность: 0.60

Трата: 'сходили в кино на выходных'
-> Категория: ЕДА
-> Уверенность: 0.49

Трата: 'заказал роллы на ужин'
-> Категория: ЕДА
-> Уверенность: 0.62

Трата: 'что-то непонятное'
-> Категория: ЕДА
-> Уверенность: 0.21



## Загрузка модели

In [31]:
import joblib

joblib.dump(pipeline, 'expense_model.joblib')

['expense_model.joblib']